# 2. Running a SWIFT zoom

We now have a box containing a small, high-resolution particle region surrounded by much heavier particles. SWIFT can build a finer cell structure around that region while keeping a cheaper background grid over the rest of the box.

In this notebook we will choose that cell geometry, plot it, and run one short zoom simulation. There is no single best choice: too few cells leave too much work in each cell, while too many cells create unnecessary tasks and overhead.

## Build SWIFT

The zoom cell implementation comes from the `zoom-swiftcon` SWIFT branch, pinned as a Git submodule in `dependencies/SWIFT`. Run the next cell to build it. You only need to do this once.

In [ ]:
%%bash
../scripts/build_swift.sh

## Choose the geometry

We will vary three things:

- **Zoom cells per axis:** how finely SWIFT divides the high-resolution region.
- **Background cells per axis:** the grid covering the full box.
- **PM mesh resolution:** the Fourier mesh used for long-range gravity.

Larger values are not automatically better. The three values appear directly in the command below. Change them there, then use the geometry plots to decide whether they look sensible before starting the simulation. By all means come back and try something different if the cell structure looks poor.

Before running the next cell, make a quick prediction. Will your choices produce lots of lightly occupied cells, or fewer heavily occupied ones? Which part of the calculation do you expect to be most expensive?

## Plot the cell geometry

`--dump-zoom-geometry` loads the particles, constructs the cells, writes a set of diagnostic files, and stops before taking a simulation step.

In [ ]:
%%bash
../scripts/run_swift.sh geometry 4 \
  ZoomRegion:zoom_min_top_level_cells:8 \
  ZoomRegion:bkg_top_level_cells:8 \
  Gravity:mesh_side_length:128

In [ ]:
from pathlib import Path
from IPython.display import Image, display

plot_dir = Path('../runs/zoom/geometry_plots')
for path in sorted(plot_dir.glob('*.png')):
    print(path.name)
    display(Image(filename=path))

Look at the cell counts and occupancy plots. Does the high-resolution region sit comfortably inside the zoom cells? Are you creating a huge number of nearly empty cells? If you want to change anything, edit the three values above and rerun the geometry cells.

## Run the simulation

We will run a fixed number of global SWIFT steps. Individual particles still use adaptive timesteps, but everyone is asking SWIFT to advance through the same number of top-level steps. The output log records the time taken, so we can later see how different geometry choices behaved.

In [ ]:
%%bash
../scripts/run_swift.sh run 4 16 \
  ZoomRegion:zoom_min_top_level_cells:8 \
  ZoomRegion:bkg_top_level_cells:8 \
  Gravity:mesh_side_length:128

Your snapshots and `output.log` are now in `runs/zoom/`. If we collect timings after the workshop, the log, `overrides.txt`, `zoom_metadata.yml`, and `zoom_cell_data.dat` are the useful files.

On COSMA8, submit `submission_scripts/swift_cosma8.slurm`. Use `sbatch --export=ALL,MODE=geometry submission_scripts/swift_cosma8.slurm` for the geometry pass, then submit it without `MODE` for the run. The job uses one threaded process on `cosma8-serial`.

Once the run is finished, you have completed the workshop.

## Optional: SWIFT-EAGLE

The main exercise is DMO. If there is time, the commands below build the corresponding SWIFT-EAGLE setup and download its cooling, yield, and photometry tables. We keep this as a separate extension rather than mixing it into the geometry exercise.

In [ ]:
%%bash
../scripts/build_swift_eagle.sh
../scripts/download_eagle_tables.sh